# Misinformation SEIR Model: Baseline vs Interventions

## Overview

This notebook demonstrates the misinformation epidemic model using deterministic SEIR dynamics.

We compare four scenarios:
1. **Baseline**: No intervention
2. **Reduced Exposure**: Lower media exposure rate
3. **Increased Recovery**: Faster fact-checking and correction
4. **Education Intervention**: Improved critical thinking and lower exposure

This analysis shows the effectiveness of each intervention in reducing misinformation spread.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.population import generate_population
from src.experiments import run_all_experiments
from src.visualization import plot_seir, plot_intervention_comparison

## Step 1: Generate Synthetic Population

Create a population with cognitive (CRT score) and behavioral (media exposure) features.

In [ ]:
# Generate synthetic population
population_size = 5000
population = generate_population(population_size, seed=42)

print(f"Population size: {len(population)}")
print(f"\nFirst few individuals:")
print(population.head(10))
print(f"\nPopulation statistics:")
print(population.describe())

## Step 2: Visualize Population Features

Explore the distribution of cognitive reflection and media exposure.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# CRT score distribution
axes[0].hist(population['crt_score'], bins=6, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('CRT Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Cognitive Reflection Test Scores')
axes[0].grid(alpha=0.3)

# Media exposure distribution
axes[1].hist(population['media_exposure'], bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[1].set_xlabel('Media Exposure (hours/day)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Daily Media Exposure')
axes[1].grid(alpha=0.3)

# Susceptibility vs features
scatter = axes[2].scatter(population['crt_score'], population['media_exposure'], 
                          c=population['susceptibility'], cmap='RdYlGn_r', s=20, alpha=0.6)
axes[2].set_xlabel('CRT Score')
axes[2].set_ylabel('Media Exposure (hours/day)')
axes[2].set_title('CRT vs Media Exposure (colored by susceptibility)')
cbar = plt.colorbar(scatter, ax=axes[2])
cbar.set_label('Susceptibility')

plt.tight_layout()
plt.show()

print(f"\nSusceptibility statistics:")
print(f"Mean susceptibility: {population['susceptibility'].mean():.3f}")
print(f"Std susceptibility: {population['susceptibility'].std():.3f}")

## Step 3: Run All Intervention Scenarios

Simulate the baseline and three intervention strategies over 180 days.

In [ ]:
# Run all experiments with the same population size and duration
print(f"Running {population_size} individual simulation over 180 days...")
print("This may take a moment.\n")

scenarios = run_all_experiments(population_size=population_size, days=180)

print(f"Completed {len(scenarios)} scenarios:")
for s in scenarios:
    print(f"  - {s['name']}")

## Step 4: Compare Peak Infections Across Scenarios

In [ ]:
# Create summary table
summary_data = []
for scenario in scenarios:
    summary_data.append({
        'Scenario': scenario['name'].replace('_', ' ').title(),
        'Beta (exposure)': f"{scenario['parameters']['beta']:.4f}",
        'Sigma (adoption)': f"{scenario['parameters']['sigma']:.4f}",
        'Peak Infection': f"{scenario['metrics']['peak_infection']:.4f}",
        'Time to Peak (days)': f"{scenario['metrics']['time_to_peak']:.1f}",
        'Final Recovered': f"{scenario['metrics']['final_recovered']:.4f}",
        'Disease Burden': f"{scenario['metrics']['area_under_infection_curve']:.4f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*150)
print("INTERVENTION COMPARISON SUMMARY")
print("="*150)
print(summary_df.to_string(index=False))
print("="*150)

## Step 5: Visualize SEIR Dynamics

Plot S, E, I, R trajectories for the baseline scenario.

In [ ]:
# Plot baseline SEIR dynamics
baseline_scenario = [s for s in scenarios if s['name'] == 'baseline'][0]
ax = plot_seir(baseline_scenario['time_series'], title='Baseline: SEIR Dynamics of Misinformation')
plt.tight_layout()
plt.show()

## Step 6: Compare Infection Trajectories Across Interventions

In [ ]:
# Plot intervention comparison for I (infected)
ax = plot_intervention_comparison(scenarios, compartment='I', title='Intervention Effectiveness: Infected (I)')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Reduced Exposure: Lower exposure rate flattens the curve significantly")
print("- Increased Recovery: Faster recovery catches infected individuals sooner")
print("- Education Intervention: Combined effect of improved cognition + lower exposure")

## Step 7: Compare Exposed (E) Compartment

In [ ]:
# Plot intervention comparison for E (exposed)
ax = plot_intervention_comparison(scenarios, compartment='E', title='Intervention Effectiveness: Exposed (E)')
plt.tight_layout()
plt.show()

print("\nThe exposed compartment shows:")
print("- Reduced Exposure: Fewer people see misinformation in the first place")
print("- Increased Recovery: Same exposure, but faster correction")
print("- Education Intervention: Most comprehensive reduction in exposure")

## Step 8: Quantify Intervention Impact

Calculate the percentage reduction in peak infection for each intervention.

In [ ]:
baseline_peak = [s['metrics']['peak_infection'] for s in scenarios if s['name'] == 'baseline'][0]

print("\nIntervention Effectiveness Analysis")
print("="*60)
print(f"Baseline Peak Infection: {baseline_peak:.4f}")
print()

for scenario in scenarios:
    if scenario['name'] != 'baseline':
        peak = scenario['metrics']['peak_infection']
        reduction = ((baseline_peak - peak) / baseline_peak) * 100
        burden_reduction = ((scenarios[0]['metrics']['area_under_infection_curve'] - 
                            scenario['metrics']['area_under_infection_curve']) / 
                           scenarios[0]['metrics']['area_under_infection_curve']) * 100
        
        print(f"{scenario['name'].replace('_', ' ').title()}:")
        print(f"  Peak Infection: {peak:.4f}")
        print(f"  Peak Reduction: {reduction:.1f}%")
        print(f"  Disease Burden Reduction: {burden_reduction:.1f}%")
        print()

## Step 9: Summary and Key Insights

### Key Findings

1. **Reduced Exposure** is effective by limiting the spread: fewer people are exposed initially, which slows transmission.

2. **Increased Recovery** accelerates correction: Even with the same exposure dynamics, faster fact-checking reduces the duration of belief.

3. **Education Intervention** is most comprehensive: By improving critical thinking AND reducing media access, it combines both mechanisms.

### Implications

- **Media literacy** and **critical thinking** training reduce susceptibility (lower sigma).
- **Platform interventions** that reduce exposure without suppression are valuable (lower beta).
- **Fact-checking and debunking** speed recovery but cannot fully offset exposure without initial barriers.
- **Multi-pronged approaches** are more effective than single interventions.

### Future Work

- Calibrate parameters against real-world misinformation datasets.
- Incorporate network structure (some individuals have more influence).
- Model competing strains of misinformation.
- Estimate cost-benefit of different intervention strategies.